In [1]:
from numbers import Real
import pandas as pd
import numpy as np
import os

from sklearn import ensemble
from sklearn import metrics
from sklearn import model_selection
from sklearn import decomposition
from sklearn.preprocessing import StandardScaler
from sklearn import pipeline
from functools import partial
from lightgbm import LGBMClassifier

import optuna


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/tabular-playground-series-nov-2021/sample_submission.csv
/kaggle/input/tabular-playground-series-nov-2021/train.csv
/kaggle/input/tabular-playground-series-nov-2021/test.csv


In [2]:
df_train = pd.read_csv('../input/tabular-playground-series-nov-2021/train.csv')
X_test = pd.read_csv('../input/tabular-playground-series-nov-2021/test.csv')
sample_submission = pd.read_csv('../input/tabular-playground-series-nov-2021/sample_submission.csv')

In [3]:
df_train.head()

,id,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f91,f92,f93,f94,f95,f96,f97,f98,f99,target
0,0,0.106643,3.59437,132.8040,3.18428,0.081971,1.18859,3.73238,2.266270,2.09959,...,1.09862,0.013331,-0.011715,0.052759,0.065400,4.211250,1.97877,0.085974,0.240496,0
1,1,0.125021,1.67336,76.5336,3.37825,0.099400,5.09366,1.27562,-0.471318,4.54594,...,3.46017,0.017054,0.124863,0.154064,0.606848,-0.267928,2.57786,-0.020877,0.024719,0
2,2,0.036330,1.49747,233.5460,2.19435,0.026914,3.12694,5.05687,3.849460,1.80187,...,4.88300,0.085222,0.032396,0.116092,-0.001688,-0.520069,2.14112,0.124464,0.148209,0
3,3,-0.014077,0.24600,779.9670,1.89064,0.006948,1.53112,2.69800,4.517330,4.50332,...,3.47439,-0.017103,-0.008100,0.062013,0.041193,0.511657,1.96860,0.040017,0.044873,0
4,4,-0.003259,3.71542,156.1280,2.14772,0.018284,2.09859,4.15492,-0.038236,3.37145,...,1.91059,-0.042943,0.105616,0.125072,0.037509,1.043790,1.07481,-0.012819,0.072798,1


In [4]:
print(df_train.shape)
print(X_test.shape)

(600000, 102)
(540000, 101)


## Dividing dependent and independent variables and adding new features
It can be seen that columns like "Id" are unique, hence wont contribute for our predictions. Therefore, these must be removed from both training and testing datasets.

In [5]:
y = df_train['target']
df_train.pop('target')
df_train.pop('id')
X_test.pop('id')
X=df_train

del df_train

In [6]:
print(X.shape)
print(X_test.shape)

(600000, 100)
(540000, 100)


In [7]:
st_scaler = StandardScaler()
X = st_scaler.fit_transform(X)
X_test = st_scaler.fit_transform(X_test)

## Optuna Search using LGBMClassifier
Important note
The following cell can be uncommented to run the hyperparameter tuning process which uses optuna method

In [8]:
"""def optimize(trial, x, y):

    boosting_type='gbdt'
    objective= 'binary'
    device_type = trial.suggest_categorical('device_tye', ['gpu'])
    max_depth = trial.suggest_int('max_depth', 3, 25)
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    min_child_weight = trial.suggest_int('min_child_weight', 0, 10)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3)
    reg_alpha = trial.suggest_uniform("lambda_l1", 0.0, 100.0)
    reg_lambda = trial.suggest_uniform("lambda_l2", 0.0, 100.0)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1)
    
    model = LGBMClassifier(boosting_type = boosting_type,
                          objective = objective,
                          device_type = device_type,
                          max_depth = max_depth,
                          n_estimators = n_estimators,
                          min_child_weight = min_child_weight,
                          learning_rate = learning_rate,
                          reg_alpha = reg_alpha,
                          reg_lambda = reg_lambda,
                          subsample = subsample,
                          colsample_bytree = colsample_bytree,
                            random_state  = 42, 
                          
                           
                          )
    

    kf = model_selection.StratifiedKFold(n_splits=5)
    accuracies = []
    for idx in kf.split(X=x, y = y):
        train_idx, test_idx = idx[0], idx[1]
        xtrain = x[train_idx]
        ytrain = y[train_idx]

        xtest = x[test_idx]
        ytest = y[test_idx]

        model.fit(xtrain, ytrain, 
                  eval_set = [(xtest, ytest)],
              early_stopping_rounds = 100,
              eval_metric = 'error',
             verbose = False)
        preds = model.predict_proba(xtest)[:,1]
        fold_acc = metrics.roc_auc_score(ytest, preds)
        accuracies.append(fold_acc)

    return -1.0*np.mean(accuracies)"""

'def optimize(trial, x, y):\n\n    boosting_type=\'gbdt\'\n    objective= \'binary\'\n    device_type = trial.suggest_categorical(\'device_tye\', [\'gpu\'])\n    max_depth = trial.suggest_int(\'max_depth\', 3, 25)\n    n_estimators = trial.suggest_int(\'n_estimators\', 100, 1000)\n    min_child_weight = trial.suggest_int(\'min_child_weight\', 0, 10)\n    learning_rate = trial.suggest_float(\'learning_rate\', 1e-3, 0.3)\n    reg_alpha = trial.suggest_uniform("lambda_l1", 0.0, 100.0)\n    reg_lambda = trial.suggest_uniform("lambda_l2", 0.0, 100.0)\n    subsample = trial.suggest_float(\'subsample\', 0.6, 1.0)\n    colsample_bytree = trial.suggest_float(\'colsample_bytree\', 0.6, 1)\n    \n    model = LGBMClassifier(boosting_type = boosting_type,\n                          objective = objective,\n                          device_type = device_type,\n                          max_depth = max_depth,\n                          n_estimators = n_estimators,\n                          min_child_

In [9]:
"""optimization_function = partial(optimize, x=X, y=y)
    
study = optuna.create_study(direction = "minimize")
study.optimize(optimization_function, n_trials=15)"""

'optimization_function = partial(optimize, x=X, y=y)\n    \nstudy = optuna.create_study(direction = "minimize")\nstudy.optimize(optimization_function, n_trials=15)'

In [10]:
#best_params_lgbmMinimize = study.best_params
#print(best_params_lgbmMinimize)
"""best_params = study.best_params
print(best_params)"""
best_params = {
 'max_depth': 9,
 'n_estimators': 982,
 'min_child_weight': 9,
 'learning_rate': 0.029350235356931986,
 'lambda_l1': 16.443464361150568,
 'lambda_l2': 54.868282901553144,
 'subsample': 0.7189486218834473,
 'colsample_bytree': 0.8770976505458908}

In [11]:
folds = model_selection.StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)
y_pred = np.zeros(len(X_test))
scores = []
for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    
    X_train, X_val = X[trn_idx], X[val_idx]
    y_train, y_val = y[trn_idx], y[val_idx]

    model =  LGBMClassifier( objective = 'binary', device_type = 'gpu',
                            max_depth = 9,
                            n_estimators = 982,
                          random_state  = 42, 
                            min_child_weight = 9,
                            learning_rate = 0.029350235356931986,
                            lambda_l1 = 16.443464361150568,
                            lambda_l2 = 54.868282901553144,
                            subsample = 0.7189486218834473,
                         colsample_bytree = 0.8770976505458908
                            
                          
                         )
   
    model.fit(X_train, y_train, eval_set = [(X_train, y_train), (X_val, y_val)], 
              verbose = False, early_stopping_rounds = 100)
    final_preds = model.predict_proba(X_val)[:,1]
    fold_score = metrics.roc_auc_score(y_val, final_preds)
    scores.append(fold_score)
    y_pred += model.predict_proba(X_test)[:,1] / folds.n_splits 

print(scores)

[LightGBM] [Warning] lambda_l1 is set=16.443464361150568, reg_alpha=0.0 will be ignored. Current value: lambda_l1=16.443464361150568
[LightGBM] [Warning] lambda_l2 is set=54.868282901553144, reg_lambda=0.0 will be ignored. Current value: lambda_l2=54.868282901553144
[LightGBM] [Warning] lambda_l1 is set=16.443464361150568, reg_alpha=0.0 will be ignored. Current value: lambda_l1=16.443464361150568
[LightGBM] [Warning] lambda_l2 is set=54.868282901553144, reg_lambda=0.0 will be ignored. Current value: lambda_l2=54.868282901553144
[LightGBM] [Warning] lambda_l1 is set=16.443464361150568, reg_alpha=0.0 will be ignored. Current value: lambda_l1=16.443464361150568
[LightGBM] [Warning] lambda_l2 is set=54.868282901553144, reg_lambda=0.0 will be ignored. Current value: lambda_l2=54.868282901553144
[LightGBM] [Warning] lambda_l1 is set=16.443464361150568, reg_alpha=0.0 will be ignored. Current value: lambda_l1=16.443464361150568
[LightGBM] [Warning] lambda_l2 is set=54.868282901553144, reg_lamb

In [12]:
sample_submission.head()

,id,target
0,600000,0.5
1,600001,0.5
2,600002,0.5
3,600003,0.5
4,600004,0.5


In [13]:
sample_submission['target'] = y_pred
sample_submission.to_csv('Submission.csv',index = False)